# LeetCode 588 — Design In-Memory File System bằng Python

- Link: https://leetcode.com/problems/design-in-memory-file-system/
- Độ khó: **Hard** (bài **Premium**)
- Chủ đề: Thiết kế (Design), Trie (cây tiền tố), Hash Map, Xử lý chuỗi, Sắp xếp

> **Ghi chú Premium:** đây là bài dành cho tài khoản Premium. Đề bài dưới đây được viết lại theo hiểu biết về bài (đặc tả API, ví dụ và ràng buộc phổ biến). Nếu bạn có tài khoản Premium, hãy đối chiếu lại câu chữ, ví dụ và ràng buộc trên LeetCode.

Skeleton trên LeetCode (Python3):

```python
class FileSystem:

    def __init__(self):

    def ls(self, path: str) -> List[str]:

    def mkdir(self, path: str) -> None:

    def addContentToFile(self, filePath: str, content: str) -> None:

    def readContentFromFile(self, filePath: str) -> str:


# Your FileSystem object will be instantiated and called as such:
# obj = FileSystem()
# param_1 = obj.ls(path)
# obj.mkdir(path)
# obj.addContentToFile(filePath,content)
# param_4 = obj.readContentFromFile(filePath)
```

## 1. Giới thiệu & mục tiêu học

Đây là bài **thiết kế lớp**: không có "thuật toán" khó, nhưng phải chọn **cấu trúc dữ liệu** phù hợp để mỗi thao tác vừa đúng vừa nhanh, và phải xử lý chuỗi đường dẫn cẩn thận. Cấu trúc tự nhiên nhất là **Trie** — mỗi nút là một thư mục/tệp, mỗi cạnh là một tên.

Sau notebook này bạn sẽ:

1. Đọc hiểu đặc tả một bài **design**: mỗi phương thức làm gì, trả về gì, có giả định gì.
2. Viết bản **"phẳng"** dùng dict đường dẫn → nội dung (dễ, làm chuẩn đối chiếu).
3. Xây dựng **Trie đường dẫn**: tách đường dẫn, đi xuống/tạo nút, phân biệt tệp và thư mục.
4. Tối ưu `ls` bằng cách giữ danh sách con **luôn được sắp xếp** (`bisect.insort`).
5. Kiểm thử bài design bằng **chuỗi thao tác ngẫu nhiên có seed**, so sánh các cài đặt với nhau.

## 2. Đề bài & ví dụ

**Phát biểu lại.** Thiết kế một hệ thống tệp trong bộ nhớ với lớp `FileSystem`:

- `FileSystem()`: khởi tạo hệ thống, chỉ có thư mục gốc `"/"`.
- `List[str] ls(String path)`:
  - Nếu `path` là **đường dẫn tệp**: trả về danh sách chỉ chứa **tên tệp** đó.
  - Nếu `path` là **thư mục**: trả về danh sách **tên** các tệp và thư mục con trực tiếp, **sắp xếp theo thứ tự từ điển**.
- `void mkdir(String path)`: tạo thư mục `path`. Nếu các thư mục trung gian chưa có thì **tạo luôn**.
- `void addContentToFile(String filePath, String content)`:
  - Nếu tệp chưa tồn tại: tạo tệp với nội dung `content`.
  - Nếu đã tồn tại: **nối thêm** `content` vào cuối nội dung cũ.
- `String readContentFromFile(String filePath)`: trả về nội dung tệp.

**Ví dụ**

```text
Input
["FileSystem", "ls", "mkdir", "addContentToFile", "ls", "readContentFromFile"]
[[], ["/"], ["/a/b/c"], ["/a/b/c/d", "hello"], ["/"], ["/a/b/c/d"]]
Output
[null, [], null, null, ["a"], "hello"]

Giải thích
FileSystem fs = new FileSystem();
fs.ls("/");                          // []  — gốc đang rỗng
fs.mkdir("/a/b/c");                  // tạo /a, /a/b, /a/b/c
fs.addContentToFile("/a/b/c/d", "hello");
fs.ls("/");                          // ["a"]
fs.readContentFromFile("/a/b/c/d");  // "hello"
```

**Ràng buộc (theo đề phổ biến)**

- `1 <= path.length, filePath.length <= 100`
- `path`, `filePath` là đường dẫn tuyệt đối, bắt đầu bằng `'/'`, không kết thúc bằng `'/'` (trừ đường dẫn gốc `"/"`).
- Tên thư mục/tệp chỉ gồm chữ cái thường; trong cùng một thư mục không có tệp và thư mục trùng tên.
- Mọi lời gọi `ls`, `readContentFromFile` đều trên đường dẫn đã tồn tại (đọc tệp thì tệp đã tồn tại).
- `1 <= content.length <= 50`
- Tối đa `300` lời gọi tổng cộng.

In [1]:
def run_ops(cls, ops, args):
    """Chạy chuỗi thao tác kiểu LeetCode, trả về list kết quả."""
    out = []
    obj = None
    for op, a in zip(ops, args):
        if op == "FileSystem":
            obj = cls()
            out.append(None)
        else:
            out.append(getattr(obj, op)(*a))
    return out

OPS = ["FileSystem", "ls", "mkdir", "addContentToFile", "ls", "readContentFromFile"]
ARGS = [[], ["/"], ["/a/b/c"], ["/a/b/c/d", "hello"], ["/"], ["/a/b/c/d"]]
EXPECTED = [None, [], None, None, ["a"], "hello"]
print(OPS)

['FileSystem', 'ls', 'mkdir', 'addContentToFile', 'ls', 'readContentFromFile']


## 3. Kiến thức nền

### 3.1. Tách đường dẫn bằng `split('/')` — cạm bẫy chuỗi rỗng

`"/a/b".split('/')` cho `['', 'a', 'b']` — phần tử đầu là chuỗi rỗng vì đường dẫn bắt đầu bằng `/`. Với gốc `"/"` ta được `['', '']`. Cách an toàn: **lọc bỏ chuỗi rỗng**.

In [2]:
for p in ["/a/b/c", "/", "/x"]:
    raw = p.split('/')
    parts = [s for s in raw if s]
    print(f"{p!r:>9}: split -> {raw}, lọc -> {parts}")

 '/a/b/c': split -> ['', 'a', 'b', 'c'], lọc -> ['a', 'b', 'c']
      '/': split -> ['', ''], lọc -> []
     '/x': split -> ['', 'x'], lọc -> ['x']


### 3.2. `str.startswith`, `rsplit` và nối đường dẫn

- `p.startswith("/a/")` kiểm tra tiền tố — nhớ thêm `/` ở cuối để `"/ab"` không bị coi là con của `"/a"`.
- `p.rsplit('/', 1)` tách **một lần từ bên phải**: lấy được tên phần tử cuối.
- `'/' + '/'.join(parts[:i])` dựng lại đường dẫn tiền tố.

In [3]:
print("/ab/c".startswith("/a"), "/ab/c".startswith("/a/"))
print("/a/b/c/d".rsplit('/', 1))
parts = ["a", "b", "c"]
print(['/' + '/'.join(parts[:i]) for i in range(1, 4)])

True False
['/a/b/c', 'd']
['/a', '/a/b', '/a/b/c']


### 3.3. Dict lồng nhau — mô hình cây

Một thư mục có thể biểu diễn là `dict` từ tên → nút con. Truy cập lồng: `root['a']['b']`. `dict.setdefault(k, v)` trả `d[k]` nếu có, nếu chưa thì gán `v` rồi trả về — rất tiện để "đi xuống, tạo nếu chưa có".

In [4]:
root = {}
cur = root
for name in ["a", "b", "c"]:
    cur = cur.setdefault(name, {})
print(root)
print(sorted(root["a"]))

{'a': {'b': {'c': {}}}}
['b']


### 3.4. Lớp với `__slots__`

`__slots__` khai báo trước các thuộc tính của object: tiết kiệm bộ nhớ và chặn lỗi gõ sai tên thuộc tính (gán `node.contnet = ...` sẽ báo lỗi ngay).

In [5]:
class Node:
    __slots__ = ("children", "is_file", "content")
    def __init__(self):
        self.children = {}
        self.is_file = False
        self.content = ""

n = Node()
try:
    n.contnet = "typo"
except AttributeError as e:
    print("bắt được lỗi gõ sai:", e)

bắt được lỗi gõ sai: 'Node' object has no attribute 'contnet' and no __dict__ for setting new attributes


### 3.5. `bisect.insort` — chèn mà vẫn giữ list sắp xếp

`insort(lst, x)` tìm vị trí bằng tìm kiếm nhị phân rồi chèn — list luôn sắp xếp, nên `ls` chỉ cần trả bản sao, không phải `sorted` lại.

In [6]:
from bisect import insort
names = []
for nm in ["delta", "alpha", "charlie", "bravo"]:
    insort(names, nm)
    print(names)

['delta']
['alpha', 'delta']
['alpha', 'charlie', 'delta']
['alpha', 'bravo', 'charlie', 'delta']


## 4. Ý tưởng nền tảng / trực giác

### 4.1. Hệ thống tệp là một cây

```text
/                 (gốc)
└── a/
    └── b/
        └── c/
            └── d   (tệp, nội dung "hello")
```

- Mỗi **thư mục** là một nút có thể có con.
- Mỗi **tệp** là một nút lá có nội dung.
- Một đường dẫn `/a/b/c/d` là **dãy tên** đi từ gốc xuống.

### 4.2. Tính tay ví dụ

| Lệnh | Cây sau lệnh | Kết quả |
|---|---|---|
| `ls("/")` | `/` | `[]` |
| `mkdir("/a/b/c")` | `/a/b/c` (tạo 3 nút) | — |
| `addContentToFile("/a/b/c/d", "hello")` | thêm lá `d` | — |
| `ls("/")` | không đổi | `["a"]` |
| `readContentFromFile("/a/b/c/d")` | không đổi | `"hello"` |
| *(thêm)* `ls("/a/b/c/d")` | không đổi | `["d"]` — đường dẫn tệp trả về tên tệp |
| *(thêm)* `addContentToFile("/a/b/c/d", "!")` | nội dung `"hello!"` | — |

### 4.3. Hai cách lưu

1. **Phẳng:** dict `đường dẫn đầy đủ → nội dung` cho tệp, và set đường dẫn thư mục. `ls` phải **quét hết** để tìm con trực tiếp.
2. **Cây (Trie):** đi theo từng tên; `ls` chỉ nhìn con của **một** nút.

In [7]:
# Mô phỏng cây bằng dict lồng nhau: thư mục = dict, tệp = str
tree = {}
def walk(path):
    parts = [s for s in path.split('/') if s]
    cur = tree
    for name in parts[:-1]:
        cur = cur.setdefault(name, {})
    return cur, (parts[-1] if parts else None)

parent, name = walk("/a/b/c/d")
parent[name] = parent.get(name, "") + "hello"
parent[name] += "!"
print(tree)

{'a': {'b': {'c': {'d': 'hello!'}}}}


## 5. Cách 1 — Baseline "phẳng": dict đường dẫn + quét khi `ls`

**Ý tưởng.**
- `files`: dict `đường dẫn tệp → nội dung`.
- `dirs`: set đường dẫn thư mục (không cần lưu gốc).
- `mkdir`: thêm **mọi tiền tố** của đường dẫn vào `dirs`.
- `addContentToFile`: thêm thư mục cha (mọi tiền tố trừ phần cuối), rồi nối nội dung.
- `ls(path)`: nếu là tệp → `[tên]`. Ngược lại quét mọi đường dẫn bắt đầu bằng `path + '/'` (hoặc `'/'` nếu là gốc), lấy **thành phần ngay sau tiền tố**, bỏ trùng, sắp xếp.

In [8]:
from typing import List

class FileSystemFlat:
    def __init__(self):
        self.files = {}          # "/a/b/c/d" -> "hello"
        self.dirs = set()        # {"/a", "/a/b", "/a/b/c"}

    @staticmethod
    def _prefixes(path):
        parts = [s for s in path.split('/') if s]
        return ['/' + '/'.join(parts[:i]) for i in range(1, len(parts) + 1)]

    def ls(self, path: str) -> List[str]:
        if path in self.files:
            return [path.rsplit('/', 1)[1]]
        prefix = '/' if path == '/' else path + '/'
        names = set()
        for p in list(self.files) + list(self.dirs):
            if p.startswith(prefix):
                names.add(p[len(prefix):].split('/')[0])
        return sorted(names)

    def mkdir(self, path: str) -> None:
        self.dirs.update(self._prefixes(path))

    def addContentToFile(self, filePath: str, content: str) -> None:
        self.dirs.update(self._prefixes(filePath)[:-1])     # thư mục cha
        self.files[filePath] = self.files.get(filePath, "") + content

    def readContentFromFile(self, filePath: str) -> str:
        return self.files[filePath]

print(run_ops(FileSystemFlat, OPS, ARGS))
assert run_ops(FileSystemFlat, OPS, ARGS) == EXPECTED

[None, [], None, None, ['a'], 'hello']


**Độ phức tạp** (gọi `P` = tổng số đường dẫn đã lưu, `L` = độ dài đường dẫn):

- `mkdir`, `addContentToFile`: `O(L^2)` (dựng `L` tiền tố, mỗi tiền tố dài tới `L`) — nhỏ.
- `readContentFromFile`: `O(L)` (băm chuỗi).
- `ls`: **`O(P · L + m log m)`** — quét mọi đường dẫn dù thư mục đang xem chỉ có vài con.

**Nút thắt:** `ls` phải quét toàn bộ. Trong hệ thống tệp thật có hàng triệu tệp, điều này không chấp nhận được. Cần cấu trúc mà **con trực tiếp của một thư mục** lấy ra ngay.

In [9]:
import time
fs = FileSystemFlat()
for i in range(3000):
    fs.addContentToFile(f"/data/x{i}/f", "z")
fs.mkdir("/small/one")
t = time.perf_counter()
for _ in range(200):
    fs.ls("/small")
print(f"Flat: 200 lần ls('/small') khi có 3000 tệp khác: {time.perf_counter() - t:.3f}s")

Flat: 200 lần ls('/small') khi có 3000 tệp khác: 0.039s


## 6. Bước đột phá

**Insight:** Đường dẫn là **dãy tên** và các đường dẫn chia sẻ **tiền tố** → dùng **Trie** (cây tiền tố) với "ký tự" là **tên thư mục**.

- Mỗi nút: `children` (dict tên → nút), `is_file`, `content`.
- Mọi thao tác đều bắt đầu bằng **"đi xuống theo đường dẫn"**: tách tên, lần lượt vào `children[name]`, **tạo nút nếu chưa có** (với `mkdir` và `addContentToFile`).

**Quy tắc nếu…thì…:**
- Nếu `ls` gặp nút **tệp** thì trả `[tên cuối của đường dẫn]`.
- Nếu `ls` gặp nút **thư mục** thì trả `sorted(children)` (hoặc danh sách con đã được giữ sắp xếp).
- Nếu `addContentToFile` gặp tệp đã có thì **nối thêm**, không ghi đè.
- Nếu đường dẫn là `"/"` thì danh sách tên **rỗng** → nút cần tìm chính là gốc.

**Invariant:** mỗi đường dẫn tồn tại tương ứng **đúng một** nút; nút tệp không có con.

## 7. Các cách tiếp theo

Notebook trình bày **3 cách**:

| # | Cách | `ls` thư mục có `m` con |
|---|---|---|
| 1 | Phẳng (mục 5) | `O(P · L + m log m)` |
| 2 | Trie + `sorted(children)` | `O(L + m log m)` |
| 3 | Trie + danh sách con luôn sắp xếp (`insort`) | `O(L + m)` |

### 7.1. Cách 2 — Trie với dict con

**Giải thích.** Một hàm trợ giúp `_walk(path)` đi từ gốc tới nút cuối, **tạo nút còn thiếu** (an toàn vì đề đảm bảo `ls`, `read` chỉ gọi trên đường dẫn đã tồn tại, còn `mkdir`/`add` thì cần tạo).

In [10]:
class TrieNode:
    __slots__ = ("children", "is_file", "content")
    def __init__(self):
        self.children = {}
        self.is_file = False
        self.content = ""

class FileSystemTrie:
    def __init__(self):
        self.root = TrieNode()

    def _walk(self, path: str) -> TrieNode:
        node = self.root
        for name in path.split('/'):
            if not name:                     # bỏ chuỗi rỗng (đầu đường dẫn, hoặc gốc "/")
                continue
            if name not in node.children:
                node.children[name] = TrieNode()
            node = node.children[name]
        return node

    def ls(self, path: str) -> List[str]:
        node = self._walk(path)
        if node.is_file:
            return [path.rsplit('/', 1)[1]]
        return sorted(node.children)

    def mkdir(self, path: str) -> None:
        self._walk(path)

    def addContentToFile(self, filePath: str, content: str) -> None:
        node = self._walk(filePath)
        node.is_file = True
        node.content += content

    def readContentFromFile(self, filePath: str) -> str:
        return self._walk(filePath).content

print(run_ops(FileSystemTrie, OPS, ARGS))
assert run_ops(FileSystemTrie, OPS, ARGS) == EXPECTED

[None, [], None, None, ['a'], 'hello']


**Đọc code:**

- `_walk`: duyệt `path.split('/')`, bỏ tên rỗng → `"/"` không tạo nút nào, trả về gốc.
- `mkdir` chỉ cần `_walk` (tạo đủ nút trên đường đi).
- `addContentToFile`: đánh dấu `is_file = True` rồi `+=` nội dung (lần đầu `content` là `""`, nên tự động "tạo tệp").
- `ls` file: `path.rsplit('/', 1)[1]` là tên cuối.
- `sorted(node.children)` — sắp **khóa** của dict (tên con).

**Code nộp LeetCode (Cách 2):**

In [11]:
class FileSystemDictTrie:
    def __init__(self):
        self.root = {"#children": {}, "#content": None}     # content None = thư mục

    def _walk(self, path):
        node = self.root
        for name in path.split('/'):
            if name:
                node = node["#children"].setdefault(name, {"#children": {}, "#content": None})
        return node

    def ls(self, path: str) -> List[str]:
        node = self._walk(path)
        if node["#content"] is not None:
            return [path.rsplit('/', 1)[1]]
        return sorted(node["#children"])

    def mkdir(self, path: str) -> None:
        self._walk(path)

    def addContentToFile(self, filePath: str, content: str) -> None:
        node = self._walk(filePath)
        node["#content"] = (node["#content"] or "") + content

    def readContentFromFile(self, filePath: str) -> str:
        return self._walk(filePath)["#content"]

assert run_ops(FileSystemDictTrie, OPS, ARGS) == EXPECTED
print("FileSystemDictTrie OK (bản dict thuần, không cần lớp nút)")

FileSystemDictTrie OK (bản dict thuần, không cần lớp nút)


(Khi nộp lên LeetCode, đổi tên lớp thành `FileSystem`; ở mục 9 có bản nộp chính thức.)

**Độ phức tạp Cách 2:** đi xuống `O(L)` cho mọi thao tác; `ls` thư mục thêm `O(m log m)` để sắp xếp `m` con; `read` / `add` thêm chi phí nối chuỗi `O(|content|)`. Space: `O(tổng độ dài tên + tổng nội dung)`.

### 7.2. Cách 3 — Trie + danh sách con luôn sắp xếp

**Giải thích.** Nếu `ls` được gọi rất nhiều lần trên thư mục lớn, việc `sorted` mỗi lần là lãng phí. Ta giữ thêm list `names` sắp xếp ở mỗi nút; khi **tạo con mới** thì `insort(names, name)`. `ls` chỉ cần `list(names)` (`O(m)` để sao chép — tránh trả về list nội bộ cho người gọi sửa).

Đánh đổi: tạo con mới tốn `O(m)` (chèn vào list) thay vì `O(1)`.

In [12]:
class SortedNode:
    __slots__ = ("children", "names", "is_file", "content")
    def __init__(self):
        self.children = {}
        self.names = []          # tên con, luôn sắp xếp
        self.is_file = False
        self.content = []        # gom các mảnh nội dung, join khi đọc

class FileSystemSorted:
    def __init__(self):
        self.root = SortedNode()

    def _walk(self, path):
        node = self.root
        for name in path.split('/'):
            if not name:
                continue
            child = node.children.get(name)
            if child is None:
                child = node.children[name] = SortedNode()
                insort(node.names, name)
            node = child
        return node

    def ls(self, path: str) -> List[str]:
        node = self._walk(path)
        return [path.rsplit('/', 1)[1]] if node.is_file else list(node.names)

    def mkdir(self, path: str) -> None:
        self._walk(path)

    def addContentToFile(self, filePath: str, content: str) -> None:
        node = self._walk(filePath)
        node.is_file = True
        node.content.append(content)

    def readContentFromFile(self, filePath: str) -> str:
        return "".join(self._walk(filePath).content)

print(run_ops(FileSystemSorted, OPS, ARGS))
assert run_ops(FileSystemSorted, OPS, ARGS) == EXPECTED

[None, [], None, None, ['a'], 'hello']


**Đọc code:**

- `child = node.children[name] = SortedNode()` — gán chuỗi: tạo nút, lưu vào dict, và giữ trong biến `child`.
- `insort(node.names, name)` chỉ chạy khi **thực sự tạo mới** → không trùng tên.
- `content` là list các mảnh: `append` là `O(1)` khấu hao, tránh tạo lại chuỗi dài mỗi lần nối; `read` thì `join`.
- `list(node.names)` trả **bản sao**.

**Độ phức tạp Cách 3:** đi xuống `O(L)`; tạo con mới `O(m)`; `ls` `O(L + m)`; `read` `O(L + |nội dung|)`; `add` `O(L + |content|)`.

So sánh nhanh thời gian `ls` trên thư mục rất lớn:

In [13]:
for cls in (FileSystemTrie, FileSystemSorted):
    fs = cls()
    for i in range(5000):
        fs.mkdir(f"/big/d{i:05d}")
    t = time.perf_counter()
    for _ in range(200):
        fs.ls("/big")
    print(f"{cls.__name__:>16}: 200 lần ls trên thư mục 5000 con: {time.perf_counter() - t:.3f}s")

  FileSystemTrie: 200 lần ls trên thư mục 5000 con: 0.007s
FileSystemSorted: 200 lần ls trên thư mục 5000 con: 0.002s


## 8. TRACE từng bước

Chạy một chuỗi thao tác dài hơn trên Cách 2 và in cây sau mỗi lệnh.

| # | Lệnh | Kết quả | Ghi chú |
|---|---|---|---|
| 1 | `mkdir("/a/b")` | — | tạo `a`, `b` |
| 2 | `addContentToFile("/a/x", "hi")` | — | tệp `x` trong `/a` |
| 3 | `ls("/a")` | `["b", "x"]` | thư mục: sắp xếp |
| 4 | `ls("/a/x")` | `["x"]` | tệp: trả về tên tệp |
| 5 | `addContentToFile("/a/x", "!!")` | — | nối thêm |
| 6 | `readContentFromFile("/a/x")` | `"hi!!"` | |
| 7 | `mkdir("/c")` | — | |
| 8 | `ls("/")` | `["a", "c"]` | |

In [14]:
def show_tree(node, name="/", indent=""):
    label = f"{name} [tệp: {node.content!r}]" if node.is_file else (name if name == "/" else name + "/")
    print(indent + label)
    for child_name in sorted(node.children):
        show_tree(node.children[child_name], child_name, indent + "    ")

def trace_fs(ops):
    fs = FileSystemTrie()
    for i, (op, *args) in enumerate(ops, 1):
        res = getattr(fs, op)(*args)
        shown = "" if res is None else f" -> {res!r}"
        print(f"#{i} {op}{tuple(args)}{shown}")
        if op in ("mkdir", "addContentToFile"):
            show_tree(fs.root, indent="     ")
    return fs

trace_ops = [("mkdir", "/a/b"), ("addContentToFile", "/a/x", "hi"), ("ls", "/a"), ("ls", "/a/x"),
             ("addContentToFile", "/a/x", "!!"), ("readContentFromFile", "/a/x"), ("mkdir", "/c"), ("ls", "/")]
fs = trace_fs(trace_ops)
assert fs.ls("/") == ["a", "c"] and fs.readContentFromFile("/a/x") == "hi!!"

#1 mkdir('/a/b',)
     /
         a/
             b/
#2 addContentToFile('/a/x', 'hi')
     /
         a/
             b/
             x [tệp: 'hi']
#3 ls('/a',) -> ['b', 'x']
#4 ls('/a/x',) -> ['x']
#5 addContentToFile('/a/x', '!!')
     /
         a/
             b/
             x [tệp: 'hi!!']
#6 readContentFromFile('/a/x',) -> 'hi!!'
#7 mkdir('/c',)
     /
         a/
             b/
             x [tệp: 'hi!!']
         c/
#8 ls('/',) -> ['a', 'c']


Trace thêm hàm `_walk`: in từng bước đi xuống, nút nào có sẵn, nút nào vừa tạo.

In [15]:
def trace_walk(fs, path):
    node = fs.root
    print(f"_walk({path!r}): parts = {[s for s in path.split('/') if s]}")
    for name in path.split('/'):
        if not name:
            continue
        status = "có sẵn" if name in node.children else "TẠO MỚI"
        node = node.children.setdefault(name, TrieNode())
        print(f"   vào '{name}' ({status})")
    print("   => nút cuối là", "tệp" if node.is_file else "thư mục")

trace_walk(fs, "/a/b/z")
trace_walk(fs, "/a/x")
trace_walk(fs, "/")

_walk('/a/b/z'): parts = ['a', 'b', 'z']
   vào 'a' (có sẵn)
   vào 'b' (có sẵn)
   vào 'z' (TẠO MỚI)
   => nút cuối là thư mục
_walk('/a/x'): parts = ['a', 'x']
   vào 'a' (có sẵn)
   vào 'x' (có sẵn)
   => nút cuối là tệp
_walk('/'): parts = []
   => nút cuối là thư mục


## 9. Code nộp LeetCode

Chọn Trie (Cách 2) — gọn, dễ đọc; nội dung lưu bằng list mảnh để nối nhanh. Tên lớp và phương thức **đúng như LeetCode**.

In [16]:
from typing import List

class FileSystem:

    class _Node:
        __slots__ = ("children", "is_file", "parts")

        def __init__(self):
            self.children = {}       # tên -> _Node
            self.is_file = False
            self.parts = []          # các mảnh nội dung (chỉ dùng khi là tệp)

    def __init__(self):
        self.root = FileSystem._Node()

    def _walk(self, path: str) -> "FileSystem._Node":
        node = self.root
        for name in path.split('/'):
            if name:                                   # bỏ chuỗi rỗng
                nxt = node.children.get(name)
                if nxt is None:
                    nxt = node.children[name] = FileSystem._Node()
                node = nxt
        return node

    def ls(self, path: str) -> List[str]:
        node = self._walk(path)
        if node.is_file:
            return [path.rsplit('/', 1)[1]]            # đường dẫn tệp -> [tên tệp]
        return sorted(node.children)                   # thư mục -> tên con đã sắp xếp

    def mkdir(self, path: str) -> None:
        self._walk(path)                               # tạo mọi thư mục còn thiếu

    def addContentToFile(self, filePath: str, content: str) -> None:
        node = self._walk(filePath)
        node.is_file = True
        node.parts.append(content)                     # nối thêm, không ghi đè

    def readContentFromFile(self, filePath: str) -> str:
        return "".join(self._walk(filePath).parts)

assert run_ops(FileSystem, OPS, ARGS) == EXPECTED
obj = FileSystem()
obj.addContentToFile("/f", "ab"); obj.addContentToFile("/f", "cd")
assert obj.readContentFromFile("/f") == "abcd" and obj.ls("/f") == ["f"] and obj.ls("/") == ["f"]
print("FileSystem OK")

FileSystem OK


## 10. Vì sao thuật toán đúng

**Ánh xạ đường dẫn ↔ nút.** `_walk` biến đường dẫn `"/n1/n2/.../nk"` thành dãy tên `[n1, ..., nk]` (bỏ chuỗi rỗng) và đi theo đúng dãy cạnh đó từ gốc. Vì dict `children` ánh xạ mỗi tên tới **duy nhất** một nút, mỗi đường dẫn tương ứng duy nhất một nút; `"/"` tương ứng gốc. Đây là invariant của Trie.

**Từng phương thức:**
- `mkdir`: `_walk` tạo mọi nút còn thiếu dọc đường → mọi thư mục trung gian tồn tại.
- `addContentToFile`: nút cuối được đánh dấu là tệp; nội dung được **nối** vào cuối danh sách mảnh, nên sau các lần gọi `c1, c2, ...` nội dung là `c1 + c2 + ...`. Lần gọi đầu tiên tạo tệp với nội dung `c1`.
- `readContentFromFile`: ghép đúng các mảnh theo thứ tự thêm.
- `ls`: nếu nút là tệp → tên cuối; nếu là thư mục → khóa của `children` chính là **tên các con trực tiếp**, sắp xếp từ điển.

**Không phá vỡ cấu trúc:** đề đảm bảo không có tệp và thư mục trùng tên trong cùng thư mục, và `ls`/`read` chỉ gọi trên đường dẫn tồn tại — nên `_walk` không bao giờ "tạo nhầm" nút trong các thao tác đọc.

## 11. Phân tích Time/Space

Gọi `L` = độ dài đường dẫn, `m` = số con của thư mục được `ls`, `P` = tổng số đường dẫn đã lưu, `C` = độ dài nội dung.

| Thao tác | Cách 1 (phẳng) | Cách 2 (Trie + sorted) | Cách 3 (Trie + insort) |
|---|---|---|---|
| `mkdir` | `O(L^2)` | `O(L)` | `O(L + m)` khi tạo con mới |
| `addContentToFile` | `O(L^2 + C)` | `O(L + C)` | `O(L + m + C)` |
| `readContentFromFile` | `O(L + C)` | `O(L + C)` | `O(L + C)` |
| `ls` (thư mục) | `O(P · L + m log m)` | `O(L + m log m)` | `O(L + m)` |
| Bộ nhớ | `O(P · L + Σ C)` | `O(tổng tên + Σ C)` | như Cách 2 + list tên |

Với tối đa 300 lời gọi, cách nào cũng chạy được; nhưng Trie là thiết kế **đúng về mặt cấu trúc** và mở rộng tốt.

## 12. Bộ test

Kiểm thử bài design: (1) ví dụ đề, (2) edge case, (3) **chuỗi thao tác ngẫu nhiên hợp lệ** chạy song song trên mọi cài đặt, so sánh từng kết quả.

In [17]:
import random

IMPLS = [FileSystemFlat, FileSystemTrie, FileSystemDictTrie, FileSystemSorted, FileSystem]

# Ví dụ đề
for cls in IMPLS:
    assert run_ops(cls, OPS, ARGS) == EXPECTED, cls.__name__

# Edge case
for cls in IMPLS:
    fs = cls()
    assert fs.ls("/") == []                                  # gốc rỗng
    fs.addContentToFile("/z", "1")                           # tệp ngay ở gốc
    fs.mkdir("/a")
    fs.mkdir("/a")                                           # mkdir lặp lại
    fs.addContentToFile("/a/b/c/f", "x")                     # thư mục cha chưa có
    assert fs.ls("/") == ["a", "z"]
    assert fs.ls("/z") == ["z"]
    assert fs.ls("/a/b/c") == ["f"]
    assert fs.ls("/a/b/c/f") == ["f"]
    fs.addContentToFile("/a/b/c/f", "yz")
    assert fs.readContentFromFile("/a/b/c/f") == "xyz"
    fs.mkdir("/ab")                                          # tên là tiền tố của tên khác
    assert fs.ls("/a") == ["b"] and fs.ls("/") == ["a", "ab", "z"]
print("Ví dụ đề + edge case OK")

Ví dụ đề + edge case OK


In [18]:
def random_session(seed, steps=120):
    """Sinh chuỗi thao tác hợp lệ; trả về list (op, args)."""
    rng = random.Random(seed)
    files, dirs = {}, {"/"}
    names = ["a", "b", "c", "d"]
    ops = []
    for _ in range(steps):
        kind = rng.choice(["mkdir", "add", "ls", "read"])
        path = "/" + "/".join(rng.choice(names) for _ in range(rng.randint(1, 3)))
        parts = path.split('/')[1:]
        prefixes = ['/' + '/'.join(parts[:i]) for i in range(1, len(parts) + 1)]
        if kind == "mkdir":
            if any(p in files for p in prefixes):            # không được đè lên tệp
                continue
            dirs.update(prefixes)
            ops.append(("mkdir", path))
        elif kind == "add":
            if any(p in files for p in prefixes[:-1]) or path in dirs:
                continue
            dirs.update(prefixes[:-1])
            files[path] = files.get(path, "") + "q"
            ops.append(("addContentToFile", path, rng.choice(["x", "yy", "zzz"])))
        elif kind == "ls":
            target = rng.choice(sorted(dirs | set(files)))
            ops.append(("ls", target))
        else:
            if files:
                ops.append(("readContentFromFile", rng.choice(sorted(files))))
    return ops

total = 0
for seed in range(150):
    ops = random_session(seed)
    objs = [cls() for cls in IMPLS]
    for op, *args in ops:
        results = [getattr(o, op)(*args) for o in objs]
        assert all(r == results[0] for r in results), (seed, op, args, results)
        total += 1
print(f"150 phiên ngẫu nhiên, {total} thao tác: mọi cài đặt cho cùng kết quả")

150 phiên ngẫu nhiên, 14186 thao tác: mọi cài đặt cho cùng kết quả


## 13. Lỗi người mới hay gặp

### Lỗi 1 — Không lọc chuỗi rỗng khi tách đường dẫn

In [19]:
class WrongSplit(FileSystemTrie):
    def _walk(self, path):
        node = self.root
        for name in path.split('/'):            # KHÔNG lọc ''
            node = node.children.setdefault(name, TrieNode())
        return node

fs = WrongSplit()
fs.mkdir("/a")
print("ls('/') sai:", fs.ls("/"), "| đúng:", ["a"])

ls('/') sai: [] | đúng: ['a']


**Vì sao sai:** `"/a".split('/') = ['', 'a']` → tạo một nút con tên `''` dưới gốc, rồi `a` nằm dưới nút `''`. Còn `"/".split('/') = ['', '']` đi xuống hai nút rỗng. Kết quả `ls("/")` trả `['']`.

**Code đúng:** bỏ qua `name` rỗng (`if name:`).

### Lỗi 2 — `ls` trên tệp trả về nội dung hoặc đường dẫn đầy đủ

In [20]:
class WrongLsFile(FileSystemTrie):
    def ls(self, path):
        node = self._walk(path)
        if node.is_file:
            return [path]                       # SAI: trả đường dẫn đầy đủ
        return sorted(node.children)

fs = WrongLsFile()
fs.addContentToFile("/a/b", "x")
print("sai:", fs.ls("/a/b"), "| đúng:", ["b"])

sai: ['/a/b'] | đúng: ['b']


**Vì sao sai:** đề yêu cầu "danh sách chỉ chứa **tên tệp**".

**Code đúng:** `[path.rsplit('/', 1)[1]]` (hoặc `path.split('/')[-1]`).

### Lỗi 3 — `addContentToFile` ghi đè thay vì nối thêm

In [21]:
class WrongOverwrite(FileSystemTrie):
    def addContentToFile(self, filePath, content):
        node = self._walk(filePath)
        node.is_file = True
        node.content = content                  # SAI: ghi đè

fs = WrongOverwrite()
fs.addContentToFile("/f", "hello"); fs.addContentToFile("/f", " world")
print("sai:", repr(fs.readContentFromFile("/f")), "| đúng:", repr("hello world"))

sai: ' world' | đúng: 'hello world'


**Vì sao sai:** đề nói nếu tệp đã tồn tại thì **append**.

**Code đúng:** `node.content += content` (hoặc list mảnh + `join`).

### Lỗi 4 — Quên sắp xếp kết quả `ls` (hoặc so tiền tố không có `/`)

In [22]:
class WrongPrefix(FileSystemFlat):
    def ls(self, path):
        if path in self.files:
            return [path.rsplit('/', 1)[1]]
        names = set()
        for p in list(self.files) + list(self.dirs):
            if p.startswith(path) and p != path:          # SAI: thiếu '/' sau path
                names.add(p[len(path):].strip('/').split('/')[0])
        return list(names)                                 # SAI: không sắp xếp

fs = WrongPrefix()
fs.mkdir("/a/z"); fs.mkdir("/a/b"); fs.mkdir("/ab")
print("sai:", fs.ls("/a"), "| đúng:", FileSystemFlat.ls(fs, "/a"))

sai: ['z', 'b'] | đúng: ['b', 'z']


**Vì sao sai:** `"/ab".startswith("/a")` là `True` → `ab` bị coi là con của `/a` (và bị cắt thành `"b"`!). Ngoài ra thứ tự `set` không xác định — đề yêu cầu thứ tự từ điển.

**Code đúng:** tiền tố `path + '/'` (gốc thì `'/'`), và `sorted(...)`.

## 14. Mẹo & nhận dạng pattern

- **Dữ liệu dạng "đường dẫn", "phân cấp", "tiền tố chung"** (hệ thống tệp, URL, tên miền, danh mục sản phẩm) → **Trie** với "ký tự" là từng thành phần.
- Bài **design**: viết một hàm trợ giúp chung (`_walk`/`_find`) để mọi phương thức dùng lại — ít bug hơn nhiều so với lặp code tách chuỗi.
- **Tách chuỗi có ký tự phân cách ở đầu** → luôn nghĩ tới phần tử rỗng.
- **Nối chuỗi nhiều lần** → gom vào list rồi `join` (tránh `O(n^2)`).
- **Kiểm thử design** = sinh chuỗi thao tác ngẫu nhiên hợp lệ + so với một cài đặt "ngây thơ" chắc chắn đúng.

## 15. Biến thể

1. **LeetCode 1166 — Design File System:** `createPath(path, value)` chỉ thành công khi thư mục cha đã tồn tại; `get(path)`.
2. **Xóa (`rm`) và di chuyển (`mv`)** đường dẫn: trên Trie là gỡ/gắn một cây con (Bài tập 3).
3. **Kích thước thư mục (`du`)**: tổng độ dài nội dung trong cây con (Bài tập 5).
4. **LeetCode 1233 — Remove Sub-Folders from the Filesystem:** Trie hoặc sắp xếp + so tiền tố có `/`.
5. **LeetCode 609 — Find Duplicate File in System:** dict nội dung → danh sách đường dẫn.

## 16. Bảng so sánh các cách

| Cách | Ý tưởng | `ls` | `mkdir`/`add` | Space | Khi nào dùng |
|---|---|---|---|---|---|
| 1. Phẳng | dict đường dẫn → nội dung, set thư mục, quét khi `ls` | `O(P·L + m log m)` | `O(L^2 + C)` | `O(P·L)` | kiểm chứng, prototype |
| 2. Trie + sorted | nút có dict con, sắp khi `ls` | `O(L + m log m)` | `O(L + C)` | `O(tổng tên)` | **nộp bài** |
| 3. Trie + insort | giữ tên con luôn sắp xếp | `O(L + m)` | `O(L + m + C)` | `O(tổng tên)` | `ls` gọi rất nhiều |

## 17. Bài tập

**Bài 1 (đoán output).** Không chạy code, cho biết kết quả từng lệnh `ls`/`read`:

```text
mkdir("/x/y"); addContentToFile("/x/a", "1"); addContentToFile("/x/a", "2");
ls("/x"); ls("/x/a"); readContentFromFile("/x/a"); ls("/x/y")
```

**Bài 2 (tính tay).** Vẽ cây Trie sau: `mkdir("/m/n")`, `addContentToFile("/m/n/p", "Q")`, `mkdir("/m/o")`, `addContentToFile("/r", "S")`. Có bao nhiêu nút (kể cả gốc)?

**Bài 3 (điền TODO).** Thêm phương thức `rm(path)` xóa một tệp hoặc cả cây thư mục (giả sử `path` tồn tại và khác `"/"`).

**Bài 4 (tự viết lại).** Tự viết lại lớp `FileSystem` bằng Trie mà không nhìn code (lớp `MyFS`).

**Bài 5 (mini challenge).** Thêm `du(path)` trả về tổng độ dài nội dung của mọi tệp trong cây con của `path` (nếu là tệp: độ dài tệp đó).

In [23]:
# Bài 3 — TODO (không gọi khi chạy notebook)
class FSWithRm(FileSystem):
    def rm(self, path: str) -> None:
        # TODO: đi tới nút CHA của path, rồi xóa children[tên cuối]
        pass

# Bài 4 — TODO
class MyFS:
    def __init__(self):
        # TODO
        pass

# Bài 5 — TODO
class FSWithDu(FileSystem):
    def du(self, path: str) -> int:
        # TODO: DFS trên cây con, cộng len(nội dung) của các tệp
        pass

## 18. Gợi ý 3 mức

**Bài 1.**
- Mức 1: `/x` có hai con: thư mục `y` và tệp `a`.
- Mức 2: `ls` trên tệp trả về tên tệp.
- Mức 3: nội dung được nối: `"1" + "2"`.

**Bài 2.**
- Mức 1: gốc có hai con `m`, `r`.
- Mức 2: `m` có hai con `n`, `o`; `n` có con `p`.
- Mức 3: đếm: gốc, m, n, p, o, r.

**Bài 3.**
- Mức 1: tách `parent_path, name = path.rsplit('/', 1)`.
- Mức 2: `parent_path` rỗng nghĩa là cha là gốc.
- Mức 3: `del self._walk(parent_path or '/').children[name]`.

**Bài 4.**
- Mức 1: lớp nút có `children`, `is_file`, nội dung.
- Mức 2: hàm `_walk` lọc chuỗi rỗng.
- Mức 3: đối chiếu với `FileSystemFlat` bằng `random_session`.

**Bài 5.**
- Mức 1: nút tệp: trả `len(nội dung)`.
- Mức 2: nút thư mục: tổng `du` của các con.
- Mức 3: viết hàm đệ quy trên nút, không phải trên đường dẫn.

## 19. Đáp án tham khảo

**Bài 1:** `ls("/x") = ["a", "y"]`, `ls("/x/a") = ["a"]`, `read = "12"`, `ls("/x/y") = []`.

**Bài 2:** 6 nút: `/`, `m`, `n`, `p` (tệp `"Q"`), `o`, `r` (tệp `"S"`).

In [24]:
fs = FileSystem()
fs.mkdir("/x/y"); fs.addContentToFile("/x/a", "1"); fs.addContentToFile("/x/a", "2")
assert fs.ls("/x") == ["a", "y"] and fs.ls("/x/a") == ["a"]
assert fs.readContentFromFile("/x/a") == "12" and fs.ls("/x/y") == []

def count_nodes(node):
    return 1 + sum(count_nodes(c) for c in node.children.values())

fs = FileSystem()
fs.mkdir("/m/n"); fs.addContentToFile("/m/n/p", "Q"); fs.mkdir("/m/o"); fs.addContentToFile("/r", "S")
assert count_nodes(fs.root) == 6

# Bài 3
class FSWithRm(FileSystem):
    def rm(self, path: str) -> None:
        parent_path, name = path.rsplit('/', 1)
        del self._walk(parent_path or '/').children[name]

fs = FSWithRm()
fs.mkdir("/a/b/c"); fs.addContentToFile("/a/f", "x")
fs.rm("/a/b")
assert fs.ls("/a") == ["f"]
fs.rm("/a/f")
assert fs.ls("/a") == []
print("Bài 1, 2, 3 OK")

Bài 1, 2, 3 OK


In [25]:
# Bài 4
class MyFS:
    def __init__(self):
        self.root = {}                      # thư mục = dict, tệp = list mảnh nội dung

    def _parent(self, path):
        parts = [s for s in path.split('/') if s]
        cur = self.root
        for name in parts[:-1]:
            cur = cur.setdefault(name, {})
        return cur, (parts[-1] if parts else None)

    def ls(self, path):
        parent, name = self._parent(path)
        if name is None:
            return sorted(self.root)
        node = parent[name]
        return [name] if isinstance(node, list) else sorted(node)

    def mkdir(self, path):
        parent, name = self._parent(path)
        if name is not None:
            parent.setdefault(name, {})

    def addContentToFile(self, filePath, content):
        parent, name = self._parent(filePath)
        parent.setdefault(name, []).append(content)

    def readContentFromFile(self, filePath):
        parent, name = self._parent(filePath)
        return "".join(parent[name])

for seed in range(60):
    a, b = MyFS(), FileSystemFlat()
    for op, *args in random_session(1000 + seed):
        assert getattr(a, op)(*args) == getattr(b, op)(*args), (seed, op, args)

# Bài 5
class FSWithDu(FileSystem):
    def du(self, path: str) -> int:
        def size(node):
            if node.is_file:
                return sum(map(len, node.parts))
            return sum(size(c) for c in node.children.values())
        return size(self._walk(path))

fs = FSWithDu()
fs.addContentToFile("/a/x", "hello"); fs.addContentToFile("/a/b/y", "abc"); fs.addContentToFile("/z", "12")
assert fs.du("/") == 10 and fs.du("/a") == 8 and fs.du("/a/b") == 3 and fs.du("/a/x") == 5
print("Bài 4, 5 OK")

Bài 4, 5 OK


## 20. Template nhớ nhanh + Checklist trước khi submit

**Template "Trie đường dẫn":**

```python
class Node:
    __slots__ = ("children", "is_file", "parts")
    def __init__(self):
        self.children, self.is_file, self.parts = {}, False, []

def _walk(self, path):
    node = self.root
    for name in path.split('/'):
        if name:
            node = node.children.setdefault(name, Node())
    return node
```

- `ls`: tệp → `[path.rsplit('/', 1)[1]]`; thư mục → `sorted(node.children)`.
- `mkdir`: `_walk(path)`.
- `add`: `node.is_file = True; node.parts.append(content)`.
- `read`: `"".join(node.parts)`.

**Câu thần chú:** *"Tách theo '/', bỏ rỗng, đi xuống tạo dần; tệp thì nối, thư mục thì sắp."*

**Checklist trước khi submit:**

- [ ] Tên lớp `FileSystem`, đúng tên 4 phương thức: `ls`, `mkdir`, `addContentToFile`, `readContentFromFile`.
- [ ] Lọc chuỗi rỗng khi `split('/')`; `"/"` trả về gốc.
- [ ] `ls` tệp → danh sách **một tên tệp**; `ls` thư mục → **sắp xếp**.
- [ ] `addContentToFile` **nối thêm**, tạo tệp (và thư mục cha) nếu chưa có.
- [ ] Không dùng biến lớp (class attribute) có thể thay đổi để lưu trạng thái — mỗi instance một `root` riêng.
- [ ] Đã thử: gốc rỗng, tệp ở gốc, tên là tiền tố của tên khác (`/a` và `/ab`).